In [ ]:
# parameters
# BINDER_FAST: set N_q=3, M_r=8 for fast cloud execution
import math as _math
N_q = 5           # Transmon Hilbert space dimension
M_r = 15          # Resonator Hilbert space dimension
omega_q = 2 * _math.pi * 5.0   # Qubit frequency (rad/GHz), 5.0 GHz
omega_r = 2 * _math.pi * 6.5   # Resonator frequency (rad/GHz), 6.5 GHz
g = 2 * _math.pi * 0.1         # Qubit-resonator coupling (rad/GHz), 100 MHz


In [ ]:
# hide
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.hamiltonians.transmon import transmon_hamiltonian
from bosonic_gates.hamiltonians.harmonic_oscillator import (
    resonator_hamiltonian,
    coupled_system_hamiltonian,
)


## Module 1c: Transmon–Cavity Dispersive Coupling

**Learning objectives:**
- Write down the Jaynes-Cummings Hamiltonian and explain the rotating-wave approximation (RWA)
- Derive the dispersive shift $\chi = g^2/\Delta$ via the Schrieffer-Wolff transformation
- Build the joint transmon-cavity Hamiltonian using library functions
- Extract $\chi$ numerically from the joint eigenspectrum and compare to the perturbative formula

---

**Convention:** Throughout this notebook the Hilbert space is ordered as **qubit $\otimes$ cavity**
(qubit is the left tensor factor, cavity is the right tensor factor).
This follows the convention of the entire repository.

**Sections:** [1 Jaynes-Cummings](#sec1) · [2 Dispersive Limit](#sec2) · [3 Joint Hamiltonian](#sec3) · [4 Computing Chi](#sec4)


<a id="sec1"></a>
## 1  The Jaynes-Cummings Model

Consider a two-level qubit (transition frequency $\omega_q$) coupled to a harmonic cavity
(frequency $\omega_r$) via a dipole interaction $g(\hat{a}+\hat{a}^\dagger)(\hat{\sigma}_- + \hat{\sigma}_+)$.
In the lab frame the full Hamiltonian is

$$\hat{H} = \frac{\omega_q}{2}\hat{\sigma}_z + \omega_r\hat{a}^\dagger\hat{a}
+ g(\hat{a}+\hat{a}^\dagger)(\hat{\sigma}_- + \hat{\sigma}_+).$$

**Rotating-wave approximation (RWA).** Moving to a frame rotating at $\omega_r$ for the cavity
and $\omega_q$ for the qubit, the interaction term becomes

$$g(\hat{a}^\dagger\hat{\sigma}_- e^{-i(\omega_q-\omega_r)t}
+ \hat{a}\hat{\sigma}_+ e^{+i(\omega_q-\omega_r)t}
+ \hat{a}\hat{\sigma}_- e^{-i(\omega_q+\omega_r)t}
+ \hat{a}^\dagger\hat{\sigma}_+ e^{+i(\omega_q+\omega_r)t}).$$

The last two terms oscillate at $\omega_q + \omega_r \sim 10$–$15\,\text{GHz}$ — far faster than
any relevant timescale — and average to zero (secular approximation).
Dropping them gives the **Jaynes-Cummings Hamiltonian**:

$$\hat{H}_{\rm JC} = \frac{\omega_q}{2}\hat{\sigma}_z + \omega_r\hat{a}^\dagger\hat{a}
+ g(\hat{a}^\dagger\hat{\sigma}_- + \hat{a}\hat{\sigma}_+).$$

The coupling term conserves excitation number: $\hat{a}^\dagger\hat{\sigma}_-$ destroys a photon
and creates a qubit excitation (absorption), while $\hat{a}\hat{\sigma}_+$ is the time-reverse.

**Dressed states (vacuum Rabi splitting).** At zero detuning $\Delta = \omega_q - \omega_r = 0$,
the eigenstates of $\hat{H}_{\rm JC}$ in the single-excitation subspace are
the symmetric and antisymmetric superpositions $|\pm\rangle = (|e,0\rangle \pm |g,1\rangle)/\sqrt{2}$
with energies $\omega_r \pm g$.
The splitting $2g$ is the **vacuum Rabi splitting**, observable as two peaks in a transmission spectrum.

**Generalization to multilevel transmon.** For a transmon with multiple levels, the
coupling generalizes to

$$g(\hat{c}\otimes\hat{a}^\dagger + \hat{c}^\dagger\otimes\hat{a}),$$

where $\hat{c} = \sum_m \sqrt{m}|m-1\rangle\langle m|$ is the transmon lowering operator.
The detuning is now level-dependent: $\Delta_m = (E_m - E_{m-1}) - \omega_r$.

> **Lab note.** *The vacuum Rabi coupling $g/2\pi$ in circuit QED ranges from $\sim 10\,\text{MHz}$
> for weakly coupled systems to $\sim 400\,\text{MHz}$ for strongly coupled ones.
> It is engineered via the geometric overlap of the qubit electric dipole moment
> and the cavity vacuum field $E_{\rm zpf}$.
> Strong coupling $g \gg (\kappa, \gamma)$ — where $\kappa$ and $\gamma$ are the cavity and qubit
> decay rates — is required for coherent quantum operations.
> This was first achieved in 3D cavities by Wallraff et al. (Nature 2004).*


<a id="sec2"></a>
## 2  The Dispersive Limit

When the qubit-cavity detuning $\Delta = \omega_q - \omega_r$ satisfies $|g/\Delta| \ll 1$,
the system is in the **dispersive regime** and direct photon exchange is suppressed.

**Schrieffer-Wolff transformation.** A unitary rotation
$U = \exp\!\left[\lambda(\hat{a}^\dagger\hat{\sigma}_- - \hat{a}\hat{\sigma}_+)\right]$
with $\lambda = g/\Delta$ block-diagonalizes $\hat{H}_{\rm JC}$ to second order in $g/\Delta$:

$$\hat{H}_{\rm disp} = \left(\omega_r + \chi\hat{\sigma}_z\right)\hat{a}^\dagger\hat{a}
+ \frac{1}{2}(\omega_q + \chi)\hat{\sigma}_z,$$

where the **dispersive shift** is

$$\chi = \frac{g^2}{\Delta} = \frac{g^2}{\omega_q - \omega_r}.$$

**Physical picture.** Rewrite as

$$\hat{H}_{\rm disp} = \omega_r'\hat{a}^\dagger\hat{a} + \frac{\tilde{\omega}_q}{2}\hat{\sigma}_z,$$

with $\omega_r' = \omega_r + \chi\hat{\sigma}_z$ qubit-state dependent.
In state $|g\rangle$: cavity resonance is $\omega_r - \chi$.
In state $|e\rangle$: cavity resonance is $\omega_r + \chi$.
The qubit **imprints its state on the cavity frequency** — this is the basis of dispersive readout.

**Sign convention.** With the definition $\Delta = \omega_q - \omega_r$:
- Red-detuned qubit ($\omega_q < \omega_r$, i.e., $\Delta < 0$): $\chi < 0$ (cavity pulled down by qubit)
- Blue-detuned qubit ($\omega_q > \omega_r$, i.e., $\Delta > 0$): $\chi > 0$

For our parameters ($\omega_q/2\pi = 5\,\text{GHz}$, $\omega_r/2\pi = 6.5\,\text{GHz}$, $\Delta < 0$),
the qubit is red-detuned and $\chi < 0$.

**Multilevel correction.** For a transmon with anharmonicity $\alpha$, the second transition
at frequency $\omega_{12} = \omega_q + \alpha$ contributes an additional term.
The full dispersive shift is

$$\chi = \frac{g^2}{\Delta} - \frac{g^2}{\Delta + \alpha},$$

where $\Delta + \alpha = \omega_{12} - \omega_r$.
For $|\alpha| \ll |\Delta|$ this reduces to $\chi \approx g^2/\Delta$, but the correction
can be significant when $g$ is large.

> **Lab note.** *$\chi/2\pi$ is measured by two-tone spectroscopy: probe the cavity transmission
> while applying a second tone. The resonator frequency shifts by $2\chi$ when the qubit is
> driven from $|g\rangle$ to $|e\rangle$.
> Typical values: $\chi/2\pi \approx 0.1$–$10\,\text{MHz}$.
> The optimal readout signal-to-noise ratio requires $\chi > \kappa$ (strong dispersive limit).
> For bosonic qubit experiments (cat, GKP), $\chi/2\pi$ of the ancilla transmon is
> typically designed to be 1–3 MHz to achieve selective number-dependent rotations (SNAP gates).*


<a id="sec3"></a>
## 3  Building the Joint Qubit-Cavity Hamiltonian

We now construct the full joint Hamiltonian using the library functions.
The Hamiltonian is assembled as:

$$\hat{H} = \hat{H}_q \otimes \mathbb{1}_r + \mathbb{1}_q \otimes \hat{H}_r
+ g(\hat{c} \otimes \hat{a}^\dagger + \hat{c}^\dagger \otimes \hat{a}),$$

where $\hat{c}$ and $\hat{c}^\dagger$ are the transmon lowering/raising operators in the truncated
Hilbert space of dimension $N_q$.

**Tensor ordering:** qubit (left factor) $\otimes$ cavity (right factor).
The joint state $|q, r\rangle = |q\rangle_q \otimes |r\rangle_r$ is indexed as the QuTiP tensor
product in this order throughout the module.

For the qubit Hamiltonian we use a truncated transmon (keeping only $N_q$ levels)
so that the full nonlinearity of the Josephson cosine is captured.
Setting $E_J$ and $E_C$ to give $\omega_{01} \approx \omega_q$ ensures the transmon resonance
matches the parameter cell value.


In [ ]:
# Build the joint qubit-cavity Hamiltonian
# Choose Ej, Ec so that omega_01 ~ omega_q
# From transmon formula: omega_01 ~ sqrt(8*Ej*Ec) - Ec ~ omega_q / (2*pi)
Ec_jc = 0.2        # GHz
omega_q_ghz = omega_q / (2 * np.pi)  # GHz
# Ej from analytic: (omega_q/(2pi) + Ec)^2 / (8*Ec)
Ej_jc = (omega_q_ghz + Ec_jc)**2 / (8 * Ec_jc)
print(f"Choosing Ej = {Ej_jc:.2f} GHz, Ec = {Ec_jc} GHz for omega_01 ~ {omega_q_ghz} GHz")

H_q = transmon_hamiltonian(Ej=Ej_jc, Ec=Ec_jc, N=N_q)
H_r = resonator_hamiltonian(w=omega_r, M=M_r)
H = coupled_system_hamiltonian(H_q, H_r, N=N_q, M=M_r, g=g)

print(f"\nHilbert space dimensions:")
print(f"  Qubit (transmon) dim = {N_q}")
print(f"  Resonator dim = {M_r}")
print(f"  Joint dim = {N_q * M_r}")
print(f"\nJoint Hamiltonian shape: {H.shape}")
print(f"Is Hermitian: {H.isherm}")

# Verify qubit frequency: eigenvalues of H_q
evals_q = H_q.eigenenergies()[:3]
omega01_actual = (evals_q[1] - evals_q[0]) / (2 * np.pi)
print(f"\nActual qubit omega_01/2pi = {omega01_actual:.4f} GHz  "
      f"(target: {omega_q_ghz:.4f} GHz)")
print(f"Detuning Delta/2pi = omega_q - omega_r = "
      f"{(omega_q - omega_r)/(2*np.pi):.4f} GHz")


<a id="sec4"></a>
## 4  Computing $\chi$ from the Eigenspectrum

We extract the dispersive shift numerically by identifying the cavity-like eigenvalues
when the qubit is in its ground state $|g\rangle$ vs excited state $|e\rangle$.

**Strategy.** In the dispersive limit, the lowest $M_r$ eigenstates of $\hat{H}$ with the qubit
in $|g\rangle$ form a dressed ladder at frequencies $\approx n\omega_r^{(g)}$, and the ladder
with the qubit in $|e\rangle$ is at $\approx n\omega_r^{(e)}$.

Concretely, we read off:

$$\omega_r^{(g)} = E_{|g,1\rangle} - E_{|g,0\rangle}, \qquad
\omega_r^{(e)} = E_{|e,1\rangle} - E_{|e,0\rangle},$$

where $|q, n\rangle$ denotes the dressed eigenstate with qubit in $q \in \{g, e\}$ and cavity in $n$.
The dispersive shift is then

$$2\chi = \omega_r^{(e)} - \omega_r^{(g)},$$

and we compare this to the perturbative prediction $\chi = g^2/(\omega_q - \omega_r)$.

We identify the dressed states by their dominant weight in the bare basis,
using the eigenvectors of $\hat{H}$.


In [ ]:
# Numerically extract chi from the joint eigenspectrum

# Diagonalize the full Hamiltonian
evals_full, evecs_full = H.eigenstates()

def find_dressed_state(evecs, evals, qubit_state, cavity_n, N_q, M_r):
    """
    Find the dressed eigenstate closest to bare product state |qubit_state, cavity_n>.
    Returns the eigenvalue index and the dressed energy.
    """
    # Build bare product state index: qubit_state * M_r + cavity_n
    bare_idx = qubit_state * M_r + cavity_n
    # Find eigenstate with maximum overlap onto this bare state
    overlaps = np.array([abs(float(evecs[i][bare_idx][0][0]))**2 for i in range(len(evecs))])
    best = np.argmax(overlaps)
    return best, evals[best]

# Find dressed energies for |g,0>, |g,1>, |e,0>, |e,1>
idx_g0, E_g0 = find_dressed_state(evecs_full, evals_full, 0, 0, N_q, M_r)
idx_g1, E_g1 = find_dressed_state(evecs_full, evals_full, 0, 1, N_q, M_r)
idx_e0, E_e0 = find_dressed_state(evecs_full, evals_full, 1, 0, N_q, M_r)
idx_e1, E_e1 = find_dressed_state(evecs_full, evals_full, 1, 1, N_q, M_r)

omega_r_g = (E_g1 - E_g0) / (2 * np.pi)  # GHz
omega_r_e = (E_e1 - E_e0) / (2 * np.pi)  # GHz
chi_numerical = (omega_r_e - omega_r_g) / 2  # GHz

# Perturbative prediction
Delta = (omega_q - omega_r)  # rad/GHz
chi_perturbative = (g**2 / Delta) / (2 * np.pi)  # GHz

# Multilevel correction (include second transmon transition)
evals_q_full = H_q.eigenenergies()[:3]
alpha_q = ((evals_q_full[2] - evals_q_full[1]) - (evals_q_full[1] - evals_q_full[0]))
Delta12 = (omega_q + alpha_q - omega_r)  # detuning for |1>->|2> transition
chi_multilevel = ((g**2 / Delta) - (g**2 / Delta12)) / (2 * np.pi)  # GHz

print("Dispersive shift extraction:")
print(f"  omega_r^(g) / 2pi = {omega_r_g:.6f} GHz  (cavity with qubit in |g>)")
print(f"  omega_r^(e) / 2pi = {omega_r_e:.6f} GHz  (cavity with qubit in |e>)")
print(f"  2*chi_numerical = {2*chi_numerical*1e3:.4f} MHz")
print(f"")
print(f"  chi_numerical        = {chi_numerical*1e3:.4f} MHz")
print(f"  chi_perturbative     = {chi_perturbative*1e3:.4f} MHz  (g^2/Delta)")
print(f"  chi_multilevel       = {chi_multilevel*1e3:.4f} MHz  (g^2/Delta - g^2/(Delta+alpha))")
print(f"")
print(f"  g / 2pi = {g/(2*np.pi)*1e3:.1f} MHz")
print(f"  Delta / 2pi = {Delta/(2*np.pi):.3f} GHz")
print(f"  g/Delta = {abs(g/Delta):.4f}  (dispersive condition g << |Delta|)")


In [ ]:
# Chi vs coupling g: compare numerical to perturbative formula
g_vals_MHz = np.linspace(10, 200, 30)   # MHz
g_vals_rad = 2 * np.pi * g_vals_MHz * 1e-3  # rad/GHz

chi_num_vals = []
chi_pert_vals = []
chi_ml_vals = []

for g_i in g_vals_rad:
    H_i = coupled_system_hamiltonian(H_q, H_r, N=N_q, M=M_r, g=g_i)
    ev_i, evec_i = H_i.eigenstates()

    _, E_g0_i = find_dressed_state(evec_i, ev_i, 0, 0, N_q, M_r)
    _, E_g1_i = find_dressed_state(evec_i, ev_i, 0, 1, N_q, M_r)
    _, E_e0_i = find_dressed_state(evec_i, ev_i, 1, 0, N_q, M_r)
    _, E_e1_i = find_dressed_state(evec_i, ev_i, 1, 1, N_q, M_r)

    chi_n = ((E_e1_i - E_e0_i) - (E_g1_i - E_g0_i)) / 2 / (2 * np.pi) * 1e3  # MHz
    chi_p = (g_i**2 / Delta) / (2 * np.pi) * 1e3  # MHz
    chi_ml_i = ((g_i**2 / Delta) - (g_i**2 / Delta12)) / (2 * np.pi) * 1e3  # MHz

    chi_num_vals.append(chi_n)
    chi_pert_vals.append(chi_p)
    chi_ml_vals.append(chi_ml_i)

chi_num_arr  = np.array(chi_num_vals)
chi_pert_arr = np.array(chi_pert_vals)
chi_ml_arr   = np.array(chi_ml_vals)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(g_vals_MHz, chi_num_arr,  lw=2.5, color="steelblue",  label="Numerical")
axes[0].plot(g_vals_MHz, chi_pert_arr, "--",   lw=2,   color="tomato",    label=r"$g^2/\Delta$")
axes[0].plot(g_vals_MHz, chi_ml_arr,   ":",    lw=2,   color="forestgreen",
             label=r"$g^2/\Delta - g^2/(\Delta+\alpha)$")
axes[0].set_xlabel(r"$g / 2\pi$ (MHz)", fontsize=12)
axes[0].set_ylabel(r"$\chi / 2\pi$ (MHz)", fontsize=12)
axes[0].set_title(r"Dispersive shift $\chi$ vs coupling $g$", fontsize=11)
axes[0].legend(fontsize=9)

# Residuals
axes[1].plot(g_vals_MHz, chi_num_arr - chi_pert_arr, lw=2, color="tomato",
             label=r"num. $-$ $g^2/\Delta$")
axes[1].plot(g_vals_MHz, chi_num_arr - chi_ml_arr, lw=2, color="forestgreen",
             label=r"num. $-$ multilevel")
axes[1].axhline(0, ls="--", color="gray", lw=1)
axes[1].set_xlabel(r"$g / 2\pi$ (MHz)", fontsize=12)
axes[1].set_ylabel(r"Residual $\Delta\chi / 2\pi$ (MHz)", fontsize=12)
axes[1].set_title("Residuals: numerical vs perturbative", fontsize=11)
axes[1].legend(fontsize=9)

plt.suptitle(
    rf"$\omega_q/2\pi = {omega_q/(2*np.pi):.1f}$ GHz, "
    rf"$\omega_r/2\pi = {omega_r/(2*np.pi):.1f}$ GHz, "
    rf"$\Delta/2\pi = {Delta/(2*np.pi):.2f}$ GHz",
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f"At g/2pi = {g/(2*np.pi)*1e3:.0f} MHz:")
print(f"  chi_numerical   = {chi_numerical*1e3:.3f} MHz")
print(f"  chi_perturbative = {chi_perturbative*1e3:.3f} MHz")
print(f"  chi_multilevel  = {chi_multilevel*1e3:.3f} MHz")
